# CKD market and evidence landscape

This notebook demonstrates three deliberately separate public-data analyses: MEPS household utilization and expenditures, CMS Medicare Part D provider/drug aggregates, and ClinicalTrials.gov registered-study activity. The source populations and grains are not interchangeable. No person-level record is joined across sources.

The repository execution uses compact, deterministic ingestion fixtures so CI is offline and reproducible. They exercise the same source contracts as the bounded official downloads documented in `docs/sources/`. Findings below are therefore fixture demonstrations, not national estimates or commercial market claims. A production refresh must replace each fixture with its own dated, checksummed source snapshot.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

from ckd_intelligence.analysis.trials import (
    summarize_trial_composition,
    summarize_trial_geography,
    summarize_trial_status,
)
from ckd_intelligence.ingestion.meps import ingest_meps
from ckd_intelligence.ingestion.partd import ingest_partd
from ckd_intelligence.ingestion.trials import ingest_trials

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FIXTURES = ROOT / 'tests' / 'fixtures' / 'ingestion'
cache = ROOT / 'data' / 'raw' / 'task5-fixture-cache'
meps = ingest_meps(FIXTURES / 'meps.csv', cache_dir=cache)
partd = ingest_partd(FIXTURES / 'partd.csv', cache_dir=cache)
trials = ingest_trials(FIXTURES / 'clinicaltrials.json', cache_dir=cache)
print({'meps_records': len(meps.valid), 'partd_records': len(partd.valid), 'trial_records': len(trials.valid), 'evidence': 'fixture_only'})

{'meps_records': 2, 'partd_records': 2, 'trial_records': 1, 'evidence': 'fixture_only'}


In [2]:
def records(batch):
    return pd.DataFrame([dict(row) for row in batch])

db = duckdb.connect()
db.execute('CREATE SCHEMA raw_meps')
db.execute('CREATE SCHEMA raw_partd')
db.execute('CREATE SCHEMA raw_trials')
db.execute('CREATE SCHEMA analytics_observed')
db.register('meps_df', records(meps.valid))
db.register('partd_df', records(partd.valid))
db.register('trials_df', records(trials.valid))
db.execute('CREATE TABLE raw_meps.events AS SELECT * FROM meps_df')
db.execute('CREATE TABLE raw_partd.records AS SELECT * FROM partd_df')
db.execute('CREATE TABLE raw_trials.studies AS SELECT * FROM trials_df')
for script in ('mart_meps_utilization.sql', 'mart_partd_prescribing.sql', 'mart_trial_landscape.sql'):
    db.execute((ROOT / 'sql' / 'marts' / script).read_text(encoding='utf-8'))
meps_mart = db.sql('SELECT * FROM analytics_observed.mart_meps_utilization').df()
partd_mart = db.sql('SELECT * FROM analytics_observed.mart_partd_prescribing').df()
trial_status = db.sql('SELECT * FROM analytics_observed.mart_trial_status').df()
display(meps_mart)
display(partd_mart)
display(trial_status)

,source,source_release,year,condition_code,event_type,observed_record_count,weighted_event_count,weighted_expenditure_usd,mean_observed_expenditure_usd,records_with_prescription_name,source_population,source_grain,evidence_type,registry_as_of_date,limitation
0,meps,HC-243-2021,2021,N18,office_visit,1,5021.4,729358.35,145.25,0.0,"U.S. civilian, noninstitutionalized population...",normalized MEPS public-use event/person-year r...,public_observed,2026-09-11,Survey weights are retained; variance design f...
1,meps,HC-243-2021,2021,I10,prescription,1,4788.1,153698.01,32.10,1.0,"U.S. civilian, noninstitutionalized population...",normalized MEPS public-use event/person-year r...,public_observed,2026-09-11,Survey weights are retained; variance design f...


,source,source_release,year,provider_npi,provider_state,generic_name,example_brand_name,total_claim_count,total_30_day_fill_count,total_drug_cost_usd,cost_per_30_day_fill_usd,is_ckd_relevant_therapy_proxy,source_population,source_grain,evidence_type,registry_as_of_date,limitation
0,partd,2024,2024,0987654321,NJ,dapagliflozin,FARXIGA,95.0,101.0,38950.0,385.643564,True,Medicare Part D prescriptions represented in C...,"provider NPI, provider state, generic drug, an...",public_observed,2026-09-11,Therapy proxy is drug-name based and not an in...
1,partd,2024,2024,1234567890,PA,empagliflozin,JARDIANCE,120.0,135.5,45678.9,337.113653,True,Medicare Part D prescriptions represented in C...,"provider NPI, provider state, generic drug, an...",public_observed,2026-09-11,Therapy proxy is drug-name based and not an in...


,source,overall_status,study_count,total_reported_enrollment,studies_with_reported_enrollment,latest_study_update_date,registry_as_of_date,evidence_type,source_grain,limitation
0,clinicaltrials,RECRUITING,1,120.0,1.0,2026-09-08,2026-09-11,public_observed,registered study grouped by recruitment status,"Registry status is submitted study metadata, n..."


In [3]:
trial_rows = [dict(row) for row in trials.valid]
status_summary = pd.DataFrame(summarize_trial_status(trial_rows))
composition_summary = pd.DataFrame(summarize_trial_composition(trial_rows))
geography_summary = pd.DataFrame(summarize_trial_geography(trial_rows))
display(status_summary)
display(composition_summary)
display(geography_summary)

,overall_status,study_count,study_share,total_reported_enrollment,studies_with_reported_enrollment,source,evidence_type,grain
0,RECRUITING,1,1.0,120,1,ClinicalTrials.gov,public_observed,registered clinical study grouped by overall_s...


,study_type,study_count,study_share,total_reported_enrollment,studies_with_reported_enrollment,source,evidence_type,grain,phase
0,INTERVENTIONAL,1,1.0,120,1,ClinicalTrials.gov,public_observed,registered clinical study grouped by study_type,NaN
1,NaN,1,1.0,120,1,ClinicalTrials.gov,public_observed,registered clinical study grouped by phase,PHASE2


,country,study_count,share_of_registered_studies,source,evidence_type,grain
0,United States,1,1.0,ClinicalTrials.gov,public_observed,registered study-country mention


## Source-specific findings and decision boundaries

- MEPS weighted event and expenditure outputs describe the survey's civilian, noninstitutionalized population. They are not provider prescribing data, and this compact fixture is not a national estimate.
- Part D outputs describe provider/drug aggregates. The CKD-relevant therapy flag is a transparent drug-name proxy only; it does not identify indication, adherence, outcomes, or beneficiaries.
- ClinicalTrials.gov outputs describe registry composition and submitted enrollment. Recruitment status and phase are not proof of active enrollment, efficacy, market share, or launch success.
- The three views can be shown together as separate evidence cards, but a combined score or patient-level join would be invalid without compatible populations, time windows, and analytic denominators.